# Single-Cell Batch Correction and Dataset Integration

**Tier 3 — Applied Bioinformatics | Module 31 · Notebook 3**

*Prerequisites: Module 30 (scRNA-seq), Notebook 2 (CITE-seq)*

---

**By the end of this notebook you will be able to:**
1. Diagnose batch effects in scRNA-seq data using mixing metrics and kBET
2. Apply Harmony, scVI, and Seurat CCA integration to correct batch effects
3. Evaluate integration quality: cell type mixing vs biological signal preservation
4. Perform label transfer between reference atlas and query datasets
5. Map query cells onto a reference atlas (Azimuth / CELLxGENE Census)


**Key resources:**
- [scib benchmarking (Luecken et al. 2022)](https://www.nature.com/articles/s41592-021-01336-8)
- [Harmony documentation](https://portals.broadinstitute.org/harmony/)
- [scVI-tools documentation](https://docs.scvi-tools.org/)
- [Azimuth reference mapping](https://azimuth.hubmapconsortium.org/)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← CITE-seq](./02_cite_seq_integration.ipynb) | [Module Overview →](../README.md)

---

## 1. Diagnosing Batch Effects

> Visual inspection: color UMAP by batch. Quantitative metrics: LISI score, kBET rejection rate, ASW (average silhouette width).

## 2. Harmony Integration

> Harmony iteratively adjusts PCA embeddings. Fast, in-memory operation. `harmonypy` Python wrapper. Visualize pre/post integration.

In [ ]:
# Example: Harmony integration
# import harmonypy as hm
# ho = hm.run_harmony(adata.obsm['X_pca'], adata.obs, 'batch')
# adata.obsm['X_pca_harmony'] = ho.Z_corr.T

## 3. scVI Deep Generative Integration

> VAE model accounting for batch as a covariate. scVI latent space for visualization and differential expression. Handles count data natively (no log normalization needed).

## 4. Label Transfer with Seurat or scANVI

> Transfer cell type annotations from labeled reference to unlabeled query. Prediction score thresholding. Validation against known markers.

## 5. Azimuth Reference Mapping

> Human Cell Atlas references (PBMC, lung, kidney). Map query cells. Interpret mapping score and uncertainty. Differential abundance testing (DAseq, Milo).